# ⛵ Sailboat Binary Classifier
Этот ноутбук предназначен для обучения модели **EfficientNet-B0** для задачи бинарной классификации: парусник или нет.

## Шаги:
1. Подготовка датасета (нарезка кропов).
2. Загрузка данных с аугментациями.
3. Обучение предобученной модели.
4. Оценка результатов на тестовой выборке.

In [ ]:
# --- ИМПОРТ ДАТАСЕТА ---
import os
from pathlib import Path

# Если вы работаете в Google Colab, разкомментируйте эти строки для загрузки с Диска:
# from google.colab import drive
# drive.mount("/content/drive")
# !cp /content/drive/MyDrive/binary_dataset.zip .
# !unzip -q binary_dataset.zip -d .

# Если вы работаете локально и у вас есть папка source_dataset:
if not Path("dataset").exists():
    print("📦 Подготовка датасета из исходных изображений...")
    !python3 prepare_data.py
else:
    print("✅ Датасет уже готов.")

In [ ]:
# Шаг 1: Подготовка данных
# Если папка dataset еще не создана, запустите этот скрипт
!python3 prepare_data.py

In [ ]:
# КОНФИГУРАЦИЯ
DATA_DIR = Path("./dataset")
BATCH_SIZE = 32
NUM_EPOCHS = 15
LEARNING_RATE = 0.001

# ПУТЬ СОХРАНЕНИЯ НА ГУГЛ ДИСК (для Colab)
IS_COLAB = "google.colab" in str(get_ipython())
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/binary_classifier_model")
else:
    SAVE_DIR = Path("./model_checkpoints")

SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"💾 Модели будут сохраняться в: {SAVE_DIR}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Используемое устройство: {device}")

### Загрузка и аугментация данных

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder(DATA_DIR / "train", train_transform)
val_ds = datasets.ImageFolder(DATA_DIR / "valid", val_test_transform)
test_ds = datasets.ImageFolder(DATA_DIR / "test", val_test_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

print(f"Классы: {train_ds.classes}")
print(f"Train: {len(train_ds)}, Valid: {len(val_ds)}, Test: {len(test_ds)}")

### Создание модели (EfficientNet-B0)

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

### Цикл обучения

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_acc = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss, train_correct = 0.0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels.data)
    
    # Валидация
    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels.data)
    
    # Статистика
    epoch_train_acc = train_correct.double().item() / len(train_ds)
    epoch_val_acc = val_correct.double().item() / len(val_ds)
    
    history["train_loss"].append(train_loss / len(train_ds))
    history["val_loss"].append(val_loss / len(val_ds))
    history["train_acc"].append(epoch_train_acc)
    history["val_acc"].append(epoch_val_acc)
    
    # СОХРАНЕНИЕ ЛУЧШЕЙ МОДЕЛИ НА ДИСК
    if epoch_val_acc > best_acc:
        best_acc = epoch_val_acc
        torch.save(model.state_dict(), SAVE_DIR / "best_model.pth")
        print(f"⭐ Найдена лучшая модель! Сохранено на Диск (Acc: {best_acc:.4f})")
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}: "
          f"Train Loss: {history["train_loss"][-1]:.4f} Acc: {epoch_train_acc:.4f} | "
          f"Val Loss: {history["val_loss"][-1]:.4f} Acc: {epoch_val_acc:.4f}")

# Финальное сохранение
torch.save(model.state_dict(), SAVE_DIR / "final_model.pth")
print("
🏁 Обучение завершено. Все модели на Google Диске!")

## Результаты обучения

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train")
plt.plot(history["val_loss"], label="Val")
plt.title("Loss Curve")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["train_acc"], label="Train")
plt.plot(history["val_acc"], label="Val")
plt.title("Accuracy Curve")
plt.legend()
plt.show()

## Финальное тестирование на Test Set

In [ ]:
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

print("
--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=train_ds.classes))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=train_ds.classes, yticklabels=train_ds.classes)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

### Визуализация предсказаний

In [ ]:
def visualize_results(model, dataset, num_images=5):
    model.eval()
    indices = random.sample(range(len(dataset)), num_images)
    plt.figure(figsize=(15, 5))
    
    for i, idx in enumerate(indices):
        img_tensor, label = dataset[idx]
        img_show = img_tensor.permute(1, 2, 0).numpy()
        img_show = np.clip(img_show * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406], 0, 1)
        
        with torch.no_grad():
            output = model(img_tensor.unsqueeze(0).to(device))
            _, pred = torch.max(output, 1)
        
        plt.subplot(1, num_images, i+1)
        plt.imshow(img_show)
        color = "green" if pred.item() == label else "red"
        plt.title(f"True: {dataset.classes[label]}
Pred: {dataset.classes[pred.item()]}", color=color)
        plt.axis("off")
    plt.show()

import random
visualize_results(model, test_ds)